In [ ]:
import pandas as pd
import os
import glob

# Folder containing the CSV files
input_folder = r"C:\Users\alexa\Thesis\OpenFace_2.2.0_win_x64\processed" # folder that has processed images and files
output_file = r"C:\Users\alexa\Thesis\combined_large.csv"

# Get a list of all CSV files in the folder
csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

# Define chunk size (adjust based on system memory)
chunk_size = 100000

# Open output file for writing
with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    write_header = True  # Ensure header is written only once

    # Loop through each CSV file
    for file_path in csv_files:
        file_name = os.path.basename(file_path)  # Extract filename for identification

        # Process CSV in chunks
        for chunk in pd.read_csv(file_path, chunksize=chunk_size):
            chunk["Source"] = file_name  # Add a column with the filename

            # Ensure "Source" is placed before "face" column
            if "face" in chunk.columns:
                cols = list(chunk.columns)
                cols.insert(cols.index("face"), cols.pop(cols.index("Source")))
                chunk = chunk[cols]

            chunk.to_csv(f_out, mode="a", index=False, header=write_header)
            write_header = False  # After first file, don't write headers again

print(f"Combined CSV saved as: {output_file}")


Data Preprocessing


In [ ]:
import pandas as pd

# Load CSV with error handling for inconsistent row lengths
file_path = r"C:\Users\alexa\Thesis\combined_large.csv"

try:
    df = pd.read_csv(file_path, engine="python", on_bad_lines="skip")  # Skips malformed rows
except Exception as e:
    print(f"Error reading CSV: {e}")
    exit()

# Clean column names (remove leading/trailing spaces)
df.columns = df.columns.str.strip()

# Ensure "Source" column is treated as text and prefixed with a space to prevent Excel issues
if "Source" in df.columns:
    df["Source"] = df["Source"].astype(str).apply(lambda x: f" {x}" if x.startswith(("-", "=")) else x)

# Save the cleaned CSV
fixed_file_path = r"C:\Users\alexa\Thesis\fixed_combined_large.csv"
df.to_csv(fixed_file_path, index=False, encoding="utf-8")

print(f"Fixed CSV saved at: {fixed_file_path}")


Data Analysis

In [ ]:
#Data Analysis
# Load the CSV
df = pd.read_csv(file_path, encoding="utf-8")

# Display the first 5 rows
print(df.head(10))

# Check column names
print(df.columns)

In [ ]:
# Check dataset size
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Check for missing values
print(df.isnull().sum())

# Get column data types
print(df.dtypes)

# Get basic statistics for numeric columns
print(df.describe())


In [ ]:
# Check unique values in the "Source" column
print(df["Source"].unique())


In [ ]:
# Show rows where a specific column (e.g., 'face') has a certain value
filtered_df = df[df["face"] > 0.5]  # Example condition
print(filtered_df.head())


In [ ]:
print(df.columns)


In [ ]:
df.columns = df.columns.str.strip().str.lower()  # Remove spaces and make lowercase
print(df.columns)


In [ ]:
import matplotlib.pyplot as plt
if "confidence" in df.columns:
    df["confidence"].hist(bins=20)
    plt.xlabel("Confidence")
    plt.ylabel("Count")
    plt.title("Confidence Distribution")
    plt.show()
else:
    print("⚠️ 'confidence' column not found!")


AU study (will be getting AU_r and AU_C)

In [ ]:
import pandas as pd

# Load the dataset
file_path = r"C:\Users\alexa\Thesis\combined_large.csv" # Update if needed
df = pd.read_csv(file_path)

# Display first few rows
print(df.head())

# Check available columns
print(df.columns)


In [ ]:
print(df.columns.tolist())  # Show all column names


In [ ]:
# Select columns that contain 'AU' in their name + keep 'Source' column
au_columns = [col for col in df.columns if 'AU' in col]
columns_to_keep = ['Source'] + au_columns  # Ensure 'Source' is included

# Filter dataset
df_filtered = df.loc[:, columns_to_keep].copy()  # Explicitly create a copy


df_filtered['Source'] = df_filtered['Source'].apply(lambda x: f"=\"{x}\"" if '=' in str(x) else x)

# Save the filtered dataset
df_filtered.to_csv("filtered_au_data.csv", index=False)

print("Filtered dataset saved as 'filtered_au_data.csv'")

df.head

In [ ]:
df.columns = df.columns.str.strip().str.replace(" ", "")
print(df.columns.tolist())  # Check if spaces were removed


In [ ]:
import pandas as pd

# Load the dataset
file_path = r"C:\Users\alexa\Thesis\filtered_au_data.csv"
df = pd.read_csv(file_path)

# Remove spaces from column names (just in case)
df.columns = df.columns.str.strip().str.replace(" ", "")

# Extract available AU columns from the dataset (excluding "Source" and "Dominant_Emotion")
available_aus = [col for col in df.columns if col.startswith("AU") and col not in ["Source", "Dominant_Emotion"]]

# Define AU to Emotion Mapping based on available AUs
emotion_map = {
    "Happiness": ["AU06_c", "AU12_c"],
    "Sadness": ["AU01_c", "AU04_c", "AU15_c"],
    "Anger": ["AU04_c", "AU05_c", "AU07_c", "AU23_c"],
    "Surprise": ["AU01_c", "AU02_c", "AU05_c", "AU26_c"],
    "Fear": ["AU01_c", "AU02_c", "AU04_c", "AU05_c", "AU20_c"],
    "Disgust": ["AU09_c", "AU15_c", "AU10_c"]  # AU10_c added since AU16_c is missing
}

# Ensure we only use AUs that exist in the dataset
for emotion in emotion_map:
    emotion_map[emotion] = [au for au in emotion_map[emotion] if au in available_aus]

# Compute summed AU intensities per source
source_au_sums = df.groupby("Source")[sum(emotion_map.values(), [])].sum()

# Map AU sums to emotion scores
emotion_scores = pd.DataFrame(index=source_au_sums.index)
for emotion, aus in emotion_map.items():
    emotion_scores[emotion] = source_au_sums[aus].sum(axis=1)  # Sum all relevant AUs per emotion

# Find the dominant emotion per source
emotion_scores["Dominant_Emotion"] = emotion_scores.idxmax(axis=1)

# Merge the dominant emotion back into the original dataset
df = df.merge(emotion_scores["Dominant_Emotion"], on="Source", how="left")

# Save the updated dataset
output_path = r"C:\Users\alexa\Thesis\filtered_au_data_with_dominant_emotion.csv"
df.to_csv(output_path, index=False)

print(f"Updated dataset saved to: {output_path}")


checking each cluster

In [ ]:
%pip install opencv-python


In [ ]:
import os

image_folder = r"C:\Users\alexa\Thesis\OpenFace_2.2.0_win_x64\processed"

# List all files in the directory
all_files = os.listdir(image_folder)

# Filter only .jpg images
jpg_images = [f for f in all_files if f.lower().endswith(".jpg")]

# Show the first 10 images
print(jpg_images[:10])

print(image_folder)



In [ ]:
import pandas as pd

# Load dataset
file_path = r"C:\Users\alexa\Thesis\filtered_au_data_with_dominant_emotion.csv"
df = pd.read_csv(file_path)

# Remove '.csv' from the Source column
df["Source"] = df["Source"].str.replace(".csv", "", regex=False)

# Save the updated file if needed
df.to_csv(file_path, index=False)

# Print first few rows to check
print(df.head())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2
import math

# Load dataset with dominant emotions
file_path = r"C:\Users\alexa\Thesis\filtered_au_data_with_dominant_emotion.csv"
df = pd.read_csv(file_path)

# Define the folder where images are stored
image_folder = r"C:\Users\alexa\Thesis\OpenFace_2.2.0_win_x64\processed"  # Change this to match your directory

# Ensure the folder exists
if not os.path.exists(image_folder):
    print(f"Image folder not found: {image_folder}")
else:
    print(f"Image folder found: {image_folder}")

# Group sources by their dominant emotion and remove duplicates
emotion_groups = df.groupby("Dominant_Emotion")["Source"].apply(lambda x: list(set(x)))  # Remove duplicates

# Define a function to display images for a given emotion dynamically
def show_images_for_emotion(emotion, sources, max_images=10):
    sources = sources[:max_images]  # Limit the number of unique images per emotion
    num_images = len(sources)

    if num_images == 0:
        print(f"No images found for {emotion}")
        return

    # Determine layout dynamically
    cols = min(num_images, 5)  # Max 5 images per row
    rows = math.ceil(num_images / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    fig.suptitle(emotion, fontsize=16)
    
    # Flatten axes for easy iteration
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    else:
        axes = axes.flatten()

    for ax, source in zip(axes, sources):
        image_path = os.path.join(image_folder, f"{source}.JPG")  # Adjust if using PNG
        if os.path.exists(image_path):
            img = cv2.imread(image_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB
            ax.imshow(img)
            ax.set_title(source, fontsize=10)
        else:
            ax.text(0.5, 0.5, "Image not found", ha="center", va="center", fontsize=12)
        ax.axis("off")

    # Hide any extra empty axes
    for ax in axes[num_images:]:
        ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for title
    plt.show()

# Show images for each emotion dynamically
for emotion, sources in emotion_groups.items():
    show_images_for_emotion(emotion, sources, max_images=50)  # Adjust max images as needed
